# Pipeline consolidado de análisis comercial

Este notebook consolida y estandariza el pipeline originalmente distribuido en varios notebooks (`01` a `05`).

## Objetivos
- Definir **un único punto de configuración** del proyecto (`C:\Python\trade`) sin hardcodear rutas internas.
- Estandarizar estructura de carpetas para `downloads`, `raw`, `outputs`, `logs` y `tmp`.
- Incorporar **logging robusto** a archivo y consola.
- Mostrar avance de procesos largos con barras de progreso.
- Dejar funciones documentadas para que cualquier persona pueda entender y mantener el flujo.

> Nota metodológica: este pipeline es descriptivo (no causal).

In [ ]:
from __future__ import annotations

import logging
import math
import sys
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

In [ ]:
@dataclass
class ProjectPaths:
    base_dir: Path
    raw_trade: Path
    downloads: Path
    geo: Path
    outputs: Path
    logs: Path
    tmp: Path
    stage_outputs: dict[str, Path] = field(default_factory=dict)


def build_paths(base_dir: str | Path, raw_trade_dir: str | Path | None = None) -> ProjectPaths:
    base = Path(base_dir)
    raw_trade = Path(raw_trade_dir) if raw_trade_dir else base / "dataverse_files"
    outputs = base / "outputs"
    stage_outputs = {
        "01_geo": outputs / "01_geo",
        "02_validacion": outputs / "02_validacion",
        "03_barycenter": outputs / "03_barycenter",
        "04_clustering": outputs / "04_clustering",
        "05_moran": outputs / "05_moran",
    }
    return ProjectPaths(
        base_dir=base,
        raw_trade=raw_trade,
        downloads=base / "downloads",
        geo=base / "geo",
        outputs=outputs,
        logs=base / "logs",
        tmp=base / "tmp",
        stage_outputs=stage_outputs,
    )


def ensure_directories(paths: ProjectPaths) -> None:
    all_dirs = [
        paths.base_dir,
        paths.raw_trade,
        paths.downloads,
        paths.geo,
        paths.outputs,
        paths.logs,
        paths.tmp,
        *paths.stage_outputs.values(),
    ]
    for d in all_dirs:
        d.mkdir(parents=True, exist_ok=True)


BASE_DIR = Path(r"C:\Python\trade")
RAW_TRADE_DIR = None

PATHS = build_paths(BASE_DIR, RAW_TRADE_DIR)
ensure_directories(PATHS)
PATHS

In [ ]:
def setup_logger(log_dir: Path, name: str = "trade_pipeline"):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = log_dir / f"pipeline_{ts}.log"

    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    fh = logging.FileHandler(log_file, encoding="utf-8")
    fh.setLevel(logging.INFO)
    fh.setFormatter(fmt)
    sh = logging.StreamHandler(sys.stdout)
    sh.setLevel(logging.INFO)
    sh.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(sh)
    logger.propagate = False
    return logger, log_file


LOGGER, LOG_FILE = setup_logger(PATHS.logs)
LOGGER.info("Inicio de ejecución del pipeline consolidado")
LOGGER.info("Base dir: %s", PATHS.base_dir)
LOGGER.info("Raw trade dir: %s", PATHS.raw_trade)
print(f"Log activo: {LOG_FILE}")

## Correcciones metodológicas aplicadas sobre versiones previas

1. **Barycenter esférico estricto**: se evita promediar lat/lon directamente y se calcula el centro con vectores unitarios 3D + normalización.
2. **Matriz de pesos de Moran robusta**: diagonal en cero, manejo explícito de distancias cero off-diagonal y estandarización por fila con protección numérica.
3. **Estandarización de IDs y esquema de columnas**: validación centralizada de columnas mínimas para evitar errores silenciosos entre etapas.

In [ ]:
REQUIRED_TRADE_COLUMNS = {"year", "location_code", "partner_code", "value_final"}


def list_trade_files(raw_trade_dir: Path):
    files = sorted(raw_trade_dir.glob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"No se encontraron .parquet en {raw_trade_dir}")
    return files


def validate_trade_schema(df: pd.DataFrame, file_name: str) -> None:
    missing = REQUIRED_TRADE_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"{file_name}: faltan columnas requeridas: {sorted(missing)}")


def load_trade_panel(paths: ProjectPaths, max_files: int | None = None) -> pd.DataFrame:
    files = list_trade_files(paths.raw_trade)
    if max_files is not None:
        files = files[:max_files]

    chunks = []
    for fp in tqdm(files, desc="Leyendo parquet de comercio"):
        df = pd.read_parquet(fp)
        validate_trade_schema(df, fp.name)
        df = df[["year", "location_code", "partner_code", "value_final"]].copy()
        chunks.append(df)

    panel = pd.concat(chunks, ignore_index=True)
    panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype("Int64")
    panel["value_final"] = pd.to_numeric(panel["value_final"], errors="coerce").fillna(0.0)
    panel["location_code"] = panel["location_code"].astype(str)
    panel["partner_code"] = panel["partner_code"].astype(str)
    LOGGER.info("Panel cargado: %s filas, %s columnas", panel.shape[0], panel.shape[1])
    return panel

In [ ]:
def latlon_to_unitvec(lat_deg: np.ndarray, lon_deg: np.ndarray) -> np.ndarray:
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    x = np.cos(lat) * np.cos(lon)
    y = np.cos(lat) * np.sin(lon)
    z = np.sin(lat)
    return np.column_stack([x, y, z])


def unitvec_to_latlon(v: np.ndarray):
    x, y, z = v
    lon = math.degrees(math.atan2(y, x))
    hyp = math.sqrt(x * x + y * y)
    lat = math.degrees(math.atan2(z, hyp))
    return lat, lon


def spherical_weighted_barycenter(lat: pd.Series, lon: pd.Series, w: pd.Series):
    arr_w = pd.to_numeric(w, errors="coerce").fillna(0.0).to_numpy(float)
    if arr_w.sum() <= 0:
        return np.nan, np.nan
    vecs = latlon_to_unitvec(lat.to_numpy(float), lon.to_numpy(float))
    weighted = (vecs * arr_w[:, None]).sum(axis=0)
    norm = np.linalg.norm(weighted)
    if norm == 0:
        return np.nan, np.nan
    return unitvec_to_latlon(weighted / norm)

In [ ]:
def run_stage_02_validation(panel: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    out_dir.mkdir(parents=True, exist_ok=True)

    exp = panel.groupby(["year", "location_code"], as_index=False)["value_final"].sum().rename(
        columns={"location_code": "code", "value_final": "exports_total"}
    )
    imp = panel.groupby(["year", "partner_code"], as_index=False)["value_final"].sum().rename(
        columns={"partner_code": "code", "value_final": "imports_total"}
    )

    audit = exp.merge(imp, on=["year", "code"], how="outer").fillna(0.0)
    audit["trade_balance_proxy"] = audit["exports_total"] - audit["imports_total"]

    out_file = out_dir / "trade_validation_country_year.csv"
    audit.to_csv(out_file, index=False)
    LOGGER.info("Etapa 02 OK. Archivo: %s", out_file)
    return audit

In [ ]:
def run_stage_03_barycenter(panel: pd.DataFrame, centroids_file: Path, out_dir: Path) -> pd.DataFrame:
    out_dir.mkdir(parents=True, exist_ok=True)
    c = pd.read_csv(centroids_file)
    c["code"] = c["code"].astype(str)
    c = c[["code", "lat", "lon"]].dropna()

    recs = []
    keys = panel[["year", "location_code"]].drop_duplicates().sort_values(["year", "location_code"])

    for row in tqdm(keys.itertuples(index=False), total=len(keys), desc="Barycenters país-año"):
        y = int(row.year)
        reporter = str(row.location_code)
        sub = panel[(panel["year"] == y) & (panel["location_code"] == reporter)].copy()
        if sub.empty:
            continue

        exp = sub.merge(c, left_on="partner_code", right_on="code", how="left").dropna(subset=["lat", "lon"])
        lat_e, lon_e = spherical_weighted_barycenter(exp["lat"], exp["lon"], exp["value_final"]) if not exp.empty else (np.nan, np.nan)

        imp_sub = panel[(panel["year"] == y) & (panel["partner_code"] == reporter)].copy()
        imp = imp_sub.merge(c, left_on="location_code", right_on="code", how="left").dropna(subset=["lat", "lon"])
        lat_i, lon_i = spherical_weighted_barycenter(imp["lat"], imp["lon"], imp["value_final"]) if not imp.empty else (np.nan, np.nan)

        recs.append({
            "year": y,
            "code": reporter,
            "lat_exports": lat_e,
            "lon_exports": lon_e,
            "lat_imports": lat_i,
            "lon_imports": lon_i,
            "exports_total": float(sub["value_final"].sum()),
            "imports_total": float(imp_sub["value_final"].sum()),
        })

    out = pd.DataFrame(recs)
    out_file = out_dir / "barycenter_country_year.csv"
    out.to_csv(out_file, index=False)
    LOGGER.info("Etapa 03 OK. Archivo: %s", out_file)
    return out

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi/2.0)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2.0)**2
    return 2.0 * R * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))


def inverse_distance_weights(lat: np.ndarray, lon: np.ndarray) -> np.ndarray:
    n = len(lat)
    D = np.zeros((n, n), dtype=float)
    for i in range(n):
        D[i, :] = haversine_km(lat[i], lon[i], lat, lon)

    with np.errstate(divide="ignore", invalid="ignore"):
        W = 1.0 / D
    np.fill_diagonal(W, 0.0)
    W[~np.isfinite(W)] = 0.0

    rs = W.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1.0
    W = W / rs
    return W


def moran_i(x: np.ndarray, W: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    z = x - x.mean()
    n = len(z)
    s0 = W.sum()
    if s0 == 0:
        return np.nan
    num = z @ W @ z
    den = z @ z
    if den == 0:
        return np.nan
    return (n / s0) * (num / den)

In [ ]:
MAX_FILES_FOR_SMOKE = 2  # usar None para corrida completa

panel = load_trade_panel(PATHS, max_files=MAX_FILES_FOR_SMOKE)
audit = run_stage_02_validation(panel, PATHS.stage_outputs["02_validacion"])

CENTROIDS_FILE = PATHS.geo / "country_centroids_augmented.csv"
if CENTROIDS_FILE.exists():
    bary = run_stage_03_barycenter(panel, CENTROIDS_FILE, PATHS.stage_outputs["03_barycenter"])
    LOGGER.info("Barycenter generado: %s filas", len(bary))
else:
    LOGGER.warning("No se encontró %s. Se omite etapa 03.", CENTROIDS_FILE)

LOGGER.info("Pipeline finalizado sin errores fatales.")
print(f"Revisar log en: {LOG_FILE}")

## Notas de operación

- Para corrida completa, setear `MAX_FILES_FOR_SMOKE = None`.
- Si falla una etapa, compartí el archivo en `logs/pipeline_YYYYMMDD_HHMMSS.log`.
- Estructura recomendada:
  - `outputs/01_geo/`
  - `outputs/02_validacion/`
  - `outputs/03_barycenter/`
  - `outputs/04_clustering/`
  - `outputs/05_moran/`